# Extract training data

This notebook will extract plate kinematic data from a plate model and other data from the `source_data` directory, writing the resulting dataset to a CSV file which can then be used to train the models in the following notebooks (`01*.ipynb`).

## Notebook setup

These cells set some of the important variables and definitions used throughout the notebook, based on the selected config file.

### Config

In [1]:
config_file = "config/.run_config.yml"

In [2]:
from lib.paths import PathConfigManager
pcm = PathConfigManager(config_file, notebook="00b")

# =====================
# Filestructure
# =====================

# Base directories
plate_model_dir = pcm.PLATE_MODEL_DIR
raster_data_dir = pcm.RASTER_DATA_DIR
mantle_data_dir = pcm.MANTLE_DATA_DIR
points_output_dir = pcm.POINTS_DATA_DIR

pcm.create_directories()

# Source data filepaths
deposits_filepath = pcm.DEPOSITS_PATH
regions_filepath = pcm.REGIONS_PATH

# Output filepaths
training_output_filepath = pcm.TRAINING_DATA_PATH

# =====================
# Notebook scope
# =====================

# Gates for determining scope of notebook run (i.e. which feature sets to extract)
use_features = pcm.use_features

# =====================
# Plate model
# =====================

# Plate model
plate_model_name = pcm.config["plate_model"]["plate_model_name"]
use_provided_plate_model = pcm.config["plate_model"]["use_provided_plate_model"]

# Timespan for analysis
min_time = pcm.config["timespan"]["min"]
max_time = pcm.config["timespan"]["max"]
times = range(min_time, max_time + 1)

# =====================
# Extraction parameters
# =====================

# Buffer distance (degrees) around reference features for sampling unlabelled points
buffer_distance = pcm.config["study_zone_buffer"]

# Number of unlabelled points to generate
num_unlabelled = pcm.config["num_unlabelled"]  # per timestep

# Random seed for reproducibility
random_seed = pcm.config["random_seed"]

# Number of processes to use
n_jobs = pcm.config["n_jobs"]

# Whether to overwrite previous rasters (default = False)
overwrite = pcm.config["overwrite_output"]

# Verbosity of logging output
verbose = pcm.config["verbose"]

### Imports

In [3]:
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    from gplately.tools import plate_isotherm_depth

from lib.assign_regions import assign_regions
from lib.calculate_convergence import run_calculate_convergence
from lib.check_files import (
    check_plate_model,
)
from lib.combine_point_data import combine_point_data
from lib.coregister_combined_point_data import run_coregister_combined_point_data
from lib.coregister_crustal_thickness import run_coregister_crustal_thickness
from lib.coregister_ocean_rasters import (
    extract_subducted_thickness,
    run_coregister_ocean_rasters,
)
from lib.create_study_area_polygons import run_create_study_area_polygons
from lib.erodep import calculate_erodep
from lib.generate_unlabelled_points import generate_unlabelled_points
from lib.misc import calculate_slab_flux, calculate_carbon
from lib.plate_models import get_plate_reconstruction
from lib.slab_dip import calculate_slab_dip
from lib.water import calculate_water_thickness
from lib.grid_features import features

# Suppress occasional joblib warnings
%env PYTHONWARNINGS=ignore::UserWarning
warnings.simplefilter("ignore", UserWarning)

objc[79746]: Class QT_ROOT_LEVEL_POOL__THESE_OBJECTS_WILL_BE_RELEASED_WHEN_QAPP_GOES_OUT_OF_SCOPE is implemented in both /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt6Core.6.11.0.dylib (0x19f963738) and /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt5Core.5.15.15.dylib (0x1943af3d0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[79746]: Class KeyValueObserver is implemented in both /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt6Core.6.11.0.dylib (0x19f963760) and /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt5Core.5.15.15.dylib (0x1943af3f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[79746]: Class RunLoopModeTracker is implemented in both /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt6Core.6.11.0.dylib (0x19f9637b0) and /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt5Core.5.15.1

env: PYTHONWARNINGS=ignore::UserWarning


/var/folders/vn/gbv894gx5fncbnm4wbnf90080000gn/T/ipykernel_79746/2198533183.py:29: UserWarning: Probing batch producer '_plate_velocity_components' raised TypeError: _plate_velocity_components() got an unexpected keyword argument 'lons'. Falling back to declares — check sampler for bugs.


### Local input and output files
If necessary, the plate model will be downloaded:

In [4]:
if use_provided_plate_model:
    check_plate_model(plate_model_dir, verbose=True)
    plate_model_name = None
plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

In [5]:
# Seafloor age grid directory
# Filename format 'seafloor_age_{time}Ma.nc'
agegrid_dir = raster_data_dir / "SeafloorAge"

# Seafloor spreading rate directory
# Filename format 'spreading_rate_{time}Ma.nc'
spreadrate_dir = raster_data_dir / "SpreadingRate"

# Seafloor sediment thickness directory
# Filename format 'sediment_thickness_{time}Ma.nc'
sedthick_dir = raster_data_dir / "SedimentThickness"

# Seafloor carbonate sediment thickness directory
# Filename format 'carbonate_thickness_{time}Ma.nc'
carbonate_dir = raster_data_dir / "CarbonateThickness"

# Oceanic crustal CO2 density directory
# Filename format 'crustal_co2_{time}Ma.nc'
co2_dir = raster_data_dir / "CrustalCO2"

# Overriding plate thickness directory
# Filename format 'crustal_thickness_{time}Ma.nc'
crustal_thickness_dir = raster_data_dir / "CrustalThickness"

# Erosion/deposition rate directory
# Filename format 'erosion_deposition_{time}Ma.nc'
erodep_dir = raster_data_dir / "ErosionDeposition"

In [6]:
# Internal file/directory paths
subduction_data_filename = points_output_dir / "subducting_plate_data.csv"
study_area_dir = points_output_dir / "study_area_polygons"
combined_points_filename = points_output_dir / "combined_points.csv"

# Cumulative training data set
coregistered_data = None

## Generate study points

### Create study area polygons along subduction zones

Here we define our study area as all points on the overriding plate within a certain distance of the subduction zone (by default, $6 \degree, \approx 660\mathrm{km}$)

In [7]:
if overwrite or not study_area_dir.is_dir():
    run_create_study_area_polygons(
        nprocs=n_jobs,
        times=times,
        plate_reconstruction=plate_model,
        output_dir=str(study_area_dir),
        buffer_distance=buffer_distance,
        verbose=verbose,
        return_output=False,
    )

### Generate random unlabelled data points

The unlabelled set is created by generating uniformly-distributed random points within the polygons created in the previous cell. To change the number of points generated at each timestep, modify the `num_unlabelled` parameter defined earlier.

In [8]:
run_generate_points = overwrite or not combined_points_filename.is_file()
if run_generate_points:
    unlabelled = generate_unlabelled_points(
        times=times,
        input_dir=study_area_dir,
        num=num_unlabelled,
        threads=n_jobs,
        seed=random_seed,
        plate_reconstruction=plate_model,
        verbose=verbose,
    )

*^^Runtime: 30.9 mins for 400 Ma*

### Combine labelled deposit/non-deposit data with random unlabelled data

The function below wrangles the points generated in the previous cell into the same format as the deposit location data.

In [9]:
combined_points = None
if run_generate_points:
    combined_points = combine_point_data(
        deposit_data=deposits_filepath,
        unlabelled_data=unlabelled,
        plate_reconstruction=plate_model,
        study_area_dir=study_area_dir,
        min_time=min(times),
        max_time=max(times),
        clip_to_study_polygons=False,  # Removing deposits outside study polygons seems cherry-picky
        n_jobs=n_jobs,
        verbose=verbose,
    )
    
    combined_points = combined_points.dropna(subset=["present_lon", "present_lat"])
    combined_points.to_csv(combined_points_filename, index=False)
    
    del unlabelled

## Feature co-registration

### Subducting plate data

This cell will extract the subduction kinematics data from the plate model, along with datasets relating to the subducting oceanic plate: seafloor age, sediment and carbonate thickness, etc.
However, if this data has already been extracted by another notebook and `overwrite` has not been set to `True`, then the data will be read from that file instead.

In [10]:
if use_features('subduction'):
    subduction_data = None
    if overwrite or not subduction_data_filename.is_file():
        subduction_data = run_calculate_convergence(
            nprocs=n_jobs,
            min_time=min(times),
            max_time=max(times),
            plate_reconstruction=plate_model,
            verbose=verbose,
        )
        subduction_data = run_coregister_ocean_rasters(
            nprocs=n_jobs,
            times=times,
            input_data=subduction_data,
            agegrid_dir=agegrid_dir,
            spreadrate_dir=spreadrate_dir,
            plate_reconstruction=plate_model,
            sedthick_dir=sedthick_dir if use_features('subduction.carbonate') else None,
            carbonate_dir=carbonate_dir if use_features('subduction.carbonate') else None,
            co2_dir=co2_dir if use_features('subduction.carbonate') else None,
            verbose=verbose,
        )
        subduction_data["plate_thickness (m)"] = plate_isotherm_depth(
            subduction_data["seafloor_age (Ma)"],
            maxiter=100,
        )
        subduction_data = calculate_slab_dip(subduction_data)
        subduction_data = calculate_slab_flux(subduction_data)

        if use_features('subduction.carbonate'):
            subduction_data = calculate_water_thickness(data=subduction_data)
            subduction_data = calculate_carbon(subduction_data)
            subduction_data = extract_subducted_thickness(
                subduction_data,
                plate_reconstruction=plate_model,
            )
            subduction_data["sediment_flux (m^2/yr)"] = (
                subduction_data["sediment_thickness (m)"]
                * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
            ).clip(0.0, np.inf)
            subduction_data["carbon_flux (t/m/yr)"] = (
                subduction_data["total_carbon_density (t/m^2)"]
                * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
            ).clip(0.0, np.inf)
            subduction_data["water_flux (m^2/yr)"] = (
                subduction_data["total_water_thickness (m)"]
                * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
            ).clip(0.0, np.inf)

        subduction_data.to_csv(subduction_data_filename, index=False)

### Assign subduction data to point deposit/non-deposit/unlabelled data

Here we assign the appropriate values for the subduction-related parameters (kinematics, seafloor age, etc.) to the deposit sites and random locations.

In [11]:
if use_features('subduction'):
    combined_points = pd.read_csv(combined_points_filename) if combined_points is None else combined_points
    subduction_data = pd.read_csv(subduction_data_filename) if subduction_data is None else subduction_data

    coregistered_data = run_coregister_combined_point_data(
        point_data=combined_points,
        subduction_data=subduction_data,
        n_jobs=n_jobs,
        verbose=verbose,
    )
    
    del combined_points, subduction_data

### Assign crustal thickness data to point data

This cell extracts the overriding plate thickness at each point.

In [12]:
if use_features('crustal'):
    coregistered_data = pd.read_csv(combined_points_filename) if coregistered_data is None else coregistered_data
    
    coregistered_data = run_coregister_crustal_thickness(
        point_data=coregistered_data,
        input_dir=crustal_thickness_dir,
        n_jobs=n_jobs,
        verbose=verbose,
    )

### Calculate cumulative erosion

Here we calculate the cumulative erosion experienced by each deposit/random point since its time of formation.

In [13]:
if use_features('erodep'):
    coregistered_data = pd.read_csv(combined_points_filename) if coregistered_data is None else coregistered_data
    
    coregistered_data = calculate_erodep(
        data = coregistered_data,
        input_dir=erodep_dir,
        n_jobs=n_jobs,
        column_name="erosion (m)",
        verbose=verbose,
    )

### Extract simple mantle features

Reconstruct labelled points to sample mantle model outputs at various depths.

In [14]:
features.available

['Base_Mantle_Features',
 'Base_Mantle_Features_LAB',
 'east_plate_velocity (cm/yr)',
 'north_plate_velocity (cm/yr)',
 'plate_acceleration (cm/yr/Myr)',
 'Temperature_Deviation_Lambdas',
 'mantle_relative_east_velocity_LAB_0km (cm/yr)',
 'mantle_relative_north_velocity_LAB_0km (cm/yr)',
 'mantle_relative_speed_LAB_0km (cm/yr)',
 'relative_velocity_parallel_to_plate_LAB_0km (cm/yr)',
 'relative_velocity_transverse_to_plate_LAB_0km (cm/yr)',
 'mantle_relative_east_velocity_LAB_40km (cm/yr)',
 'mantle_relative_north_velocity_LAB_40km (cm/yr)',
 'mantle_relative_speed_LAB_40km (cm/yr)',
 'relative_velocity_parallel_to_plate_LAB_40km (cm/yr)',
 'relative_velocity_transverse_to_plate_LAB_40km (cm/yr)',
 'mantle_relative_east_velocity_LAB_80km (cm/yr)',
 'mantle_relative_north_velocity_LAB_80km (cm/yr)',
 'mantle_relative_speed_LAB_80km (cm/yr)',
 'relative_velocity_parallel_to_plate_LAB_80km (cm/yr)',
 'relative_velocity_transverse_to_plate_LAB_80km (cm/yr)',
 'mantle_relative_east_velocity

In [15]:
# Refresh sample mantle module to reflect recent edits
import importlib
import lib.grid_features as gf
importlib.reload(gf)

features = gf.features

In [16]:
if use_features('mantle'):
    coregistered_data = pd.read_csv(combined_points_filename) if coregistered_data is None else coregistered_data

    coregistered_data = features.extract(
        point_data=coregistered_data,
        mantle_data_dir=mantle_data_dir,
        plate_reconstruction=plate_model,
    )

/Users/glados/opt/anaconda3/envs/prospectivity/lib/python3.13/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
/Users/glados/Documents/Not Useless/Documents/University/2026/Honours/Data & Code/PUB-framework-Alfonso/lib/mantle_variables.py:208: RuntimeWarning: divide by zero encountered in divide
/Users/glados/opt/anaconda3/envs/prospectivity/lib/python3.13/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
/Users/glados/opt/anaconda3/envs/prospectivity/lib/python3.13/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
/Users/glados/opt/anaconda3/envs/prospectivity/lib/python3.13/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
/Users/glados/opt/anaconda3/envs/prospectivity/lib/python3.13/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
/Users/glados/opt/anaconda3/envs/prospectivity/lib/pyt

*^^ Runtime: 14:30 for 350Ma*

In [17]:
features._results

,Radial_Velocity_100km,Radial_Velocity_200km,Radial_Velocity_300km,Radial_Velocity_400km,Temperature_Deviation_CG_100km,Temperature_Deviation_CG_200km,Temperature_Deviation_CG_300km,Temperature_Deviation_CG_400km,Speed_100km,Speed_200km,...,Temperature_Deviation_Avg_LAB-300km_Rolling_30Ma_delta,Temperature_Deviation_Avg_LAB-300km_Rolling_50Ma_delta,Temperature_Deviation_Avg_LAB-400km_delta,Temperature_Deviation_Avg_LAB-400km_Rolling_30Ma_delta,Temperature_Deviation_Avg_LAB-400km_Rolling_50Ma_delta,temperature_deviation_cg_lambda_4_(K)_delta,temperature_deviation_cg_lambda_3_(K)_delta,temperature_deviation_cg_lambda_2_(K)_delta,temperature_deviation_cg_lambda_1_(K)_delta,temperature_deviation_cg_lambda_0_(K)_delta
0,-2.893272e-13,2.073483e-11,1.941285e-10,2.248769e-10,-322.189675,-210.286253,-26.213367,-7.794119,5.858045e-10,3.806048e-10,...,2.355958,-8.192775,48.764854,2.121035,-10.242055,-1.308617e-08,0.000016,-0.005645,0.667243,-2.202600
1,-8.466822e-13,7.064558e-11,1.255349e-10,1.079810e-10,-178.843517,-79.825872,41.863143,28.690799,5.150327e-10,7.181155e-10,...,1.238922,-5.674094,2.389439,1.781965,-5.262675,1.320305e-08,-0.000006,-0.000574,0.360964,-2.520877
2,-8.466822e-13,7.064558e-11,1.255349e-10,1.079810e-10,-178.843517,-79.825872,41.863143,28.690799,5.150327e-10,7.181155e-10,...,1.238922,-5.674094,2.389439,1.781965,-5.262675,1.320305e-08,-0.000006,-0.000574,0.360964,-2.520877
3,-9.624497e-13,8.516506e-11,1.301599e-10,1.033744e-10,-173.943241,-69.025374,43.376323,29.253382,5.151739e-10,7.327527e-10,...,1.568232,-5.369569,6.464962,1.678159,-5.221725,1.951697e-08,-0.000011,0.000563,0.300616,-2.006431
4,-9.624497e-13,8.516506e-11,1.301599e-10,1.033744e-10,-173.943241,-69.025374,43.376323,29.253382,5.151739e-10,7.327527e-10,...,1.568232,-5.369569,6.464962,1.678159,-5.221725,1.951697e-08,-0.000011,0.000563,0.300616,-2.006431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45171,-5.927679e-12,-8.693295e-11,-3.201985e-10,-4.925899e-10,-185.022851,-89.878638,-44.614204,-73.038308,1.954642e-09,1.865590e-09,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
45172,-1.142061e-11,-7.990011e-11,-4.810381e-10,-7.161267e-10,-283.969259,-261.919245,-208.851841,-205.913378,1.007495e-09,1.036162e-09,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
45173,-2.618598e-11,-2.495346e-10,-8.134660e-10,-1.017621e-09,-288.471563,-356.627097,-374.695531,-299.556326,1.000780e-09,7.206821e-10,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
45174,6.813001e-13,1.288312e-10,3.336324e-10,4.087479e-10,-97.743759,18.677484,54.690760,50.122202,1.425285e-09,1.114358e-09,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000


*^^ Runtime: ~6mins for 350 Ma*

In [18]:
coregistered_data.columns.size

138

## Post-processing

### Assign data to regions

To divide the data into individual regions for the later analysis, we use the `regions_filename` defined earlier, if desired.

In [19]:
if regions_filepath is not None and regions_filepath.is_file():
    points = gpd.GeoSeries.from_xy(
        coregistered_data["present_lon"],
        coregistered_data["present_lat"],
        index=coregistered_data.index,
    )
    coregistered_data["region"] = assign_regions(
        points,
        regions=regions_filepath,
    )
    del points

### Save to file

Finally, we write the dataset to a CSV file.

In [20]:
coregistered_data.to_csv(training_output_filepath, index=False)

coregistered_data.groupby(["region", "label"]).size()

region          label     
East Asia       negative        14
                positive       159
                unlabelled    7729
North America   negative        61
                positive       293
                unlabelled    8104
Other           negative       214
                positive         2
                unlabelled    6476
South America   negative      1389
                positive       275
                unlabelled    7845
Southeast Asia  negative         4
                positive       145
                unlabelled    5457
Tethys          negative        21
                positive       481
                unlabelled    6507
dtype: int64